# SPECTRA Tutorial: Running on Rubin Science Platform (RSP)

This notebook demonstrates how to:
1. Query Rubin/LSST photometry via TAP
2. Run SPECTRA SED fitting
3. Analyze results and generate plots

**Requirements:**
- Rubin Science Platform account
- RSP authentication token
- SPECTRA installed (see [INSTALL.md](../docs/INSTALL.md))

---

## 1. Setup

In [ ]:
# Standard imports
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

# Add SPECTRA to path
spectra_path = Path.home() / "SPECTRA"  # Adjust if installed elsewhere
sys.path.insert(0, str(spectra_path))

# SPECTRA imports
from src.models.ssp_model import SSPModel
from src.likelihood import Likelihood
from src.fit import SEDFitter
from src.mcmc.mcmc_runner import MCMCRunner
from src.utils.plotting import Plotting

print("✓ SPECTRA imported successfully")

In [ ]:
# Set RSP token (get from https://data.lsst.cloud)
os.environ['RSP_TOKEN'] = "your_token_here"

# Or load from file
# with open(Path.home() / ".rsp_token", "r") as f:
#     os.environ['RSP_TOKEN'] = f.read().strip()

print("✓ RSP token set")

## 2. Query Rubin Photometry

Let's query photometry for a known galaxy.

In [ ]:
from src.data.rubin_query import RubinDataQuery

# Initialize query interface
rubin = RubinDataQuery(token=os.environ['RSP_TOKEN'])

# Example: Query by object ID
object_id = 1234567890  # Replace with real Rubin objectId

# Or query by coordinates (RA/Dec in degrees)
# ra, dec = 150.1234, 2.3456
# phot_data = rubin.query_by_coords(ra, dec, radius_arcsec=1.0)

try:
    phot_data = rubin.query_by_id(object_id)
    print(f"✓ Retrieved photometry for objectId {object_id}")
    print(f"  Bands: {phot_data['bands']}")
    print(f"  Wavelengths: {phot_data['wavelength']} Å")
    print(f"  Fluxes: {phot_data['obs_flux']} Jy")
except Exception as e:
    print(f"✗ Query failed: {e}")
    print("  Using mock data for demonstration...")
    
    # Create synthetic data for testing
    phot_data = {
        'wavelength': np.array([3557, 4825, 6223, 7546, 8691, 9712]),  # ugrizy (Å)
        'obs_flux': np.array([2.1e-6, 3.5e-6, 4.2e-6, 3.8e-6, 3.5e-6, 3.2e-6]),  # Jy
        'obs_err': np.array([2.1e-7, 2.5e-7, 2.8e-7, 2.6e-7, 2.5e-7, 2.4e-7]),
        'bands': ['u', 'g', 'r', 'i', 'z', 'y'],
        'redshift': 0.05,
        'ra': 150.1234,
        'dec': 2.3456
    }

In [ ]:
# Visualize the photometry
fig, ax = plt.subplots(figsize=(10, 5))

wav_um = phot_data['wavelength'] / 1e4  # Å → μm

ax.errorbar(wav_um, phot_data['obs_flux'], yerr=phot_data['obs_err'],
            fmt='o', markersize=8, capsize=5, label='Rubin photometry')

ax.set_xlabel('Wavelength [μm]', fontsize=12)
ax.set_ylabel('Flux [Jy]', fontsize=12)
ax.set_title('Observed SED', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## 3. Run SPECTRA Fit

Now let's fit an SSP model to the data.

In [ ]:
# Initialize SSP model
ssp_config = {
    'type': 'fsps',  # Will fall back to mock if FSPS not installed
    'redshift': phot_data.get('redshift', 0.0),
    'imf': 'chabrier',
    'dust_type': 2,  # Calzetti
    'add_neb_emission': False
}

model = SSPModel(ssp_config)
print(f"✓ SSP model initialized (mock_mode={model.mock_mode})")

# Calibrate to observed flux
model.set_flux_calibration(phot_data['obs_flux'], wavelengths=phot_data['wavelength'])

In [ ]:
# Define priors
priors = {
    'mass': [8.0, 12.0],        # log10(M/Msun) for galaxies
    'age': [0.1, 13.5],         # Gyr
    'metallicity': [-2.0, 0.5], # log10(Z/Zsun)
    'dust': [0.0, 3.0]          # E(B-V)
}

# Initialize likelihood
likelihood = Likelihood(
    obs_flux=phot_data['obs_flux'],
    obs_err=phot_data['obs_err'],
    priors=priors,
    error_floor=0.05
)
print("✓ Likelihood initialized")

### 3a. Fast Maximum Likelihood Fit

In [ ]:
%%time

# Initialize fitter
fitter = SEDFitter(likelihood, model, wavelengths=phot_data['wavelength'])

# Starting guess (midpoint of priors)
initial_params = {
    'mass': 10.0,
    'age': 3.0,
    'metallicity': -0.5,
    'dust': 0.5
}

bounds = [(pmin, pmax) for pmin, pmax in priors.values()]

# Run ML fit
results_ml = fitter.fit_maximum_likelihood(initial_params, bounds)

print("\n✓ ML Fit Results:")
for param, value in results_ml['parameters'].items():
    print(f"  {param}: {value:.3f}")
print(f"  Log-likelihood: {results_ml['log_likelihood']:.2f}")

### 3b. Full MCMC Fit (Optional - takes ~2 minutes)

In [ ]:
# Uncomment to run MCMC
# mcmc_config = {
#     'n_walkers': 32,
#     'n_steps': 1000,
#     'n_burnin': 200,
#     'thin': 5
# }
# 
# mcmc_runner = MCMCRunner(likelihood, model, config=mcmc_config)
# mcmc_runner.set_wavelengths(phot_data['wavelength'])
# 
# param_names = list(initial_params.keys())
# results_mcmc = mcmc_runner.run(initial_params, bounds, param_names)
# 
# print("\n✓ MCMC Fit Results:")
# for param, stats in results_mcmc['percentiles'].items():
#     print(f"  {param}: {stats['median']:.3f} +{stats['upper']:.3f} -{stats['lower']:.3f}")

## 4. Visualize Results

In [ ]:
# Plot SED with best-fit model
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), 
                                gridspec_kw={'height_ratios': [3, 1], 'hspace': 0})

# Top panel: SED
wav_um = phot_data['wavelength'] / 1e4

# Observed photometry
ax1.errorbar(wav_um, phot_data['obs_flux'], yerr=phot_data['obs_err'],
             fmt='o', markersize=10, capsize=4, label='Observed',
             color='#2980B9', ecolor='#2980B9')

# Model photometry
ax1.scatter(wav_um, results_ml['mod_flux'], s=120, marker='s',
            facecolor='#E74C3C', edgecolors='darkred', linewidth=1.5,
            label='Model', zorder=5)

# Smooth model spectrum
wav_smooth_aa = np.logspace(np.log10(3000), np.log10(10000), 200)
flux_smooth = model.get_magnitudes(
    wavelengths=wav_smooth_aa,
    **results_ml['parameters']
)
ax1.plot(wav_smooth_aa/1e4, flux_smooth, '-', color='#E74C3C', 
         alpha=0.7, linewidth=1.5, label='Model spectrum')

ax1.set_xscale('log')
ax1.set_yscale('log')
ax1.set_ylabel('Flux [Jy]', fontsize=12)
ax1.legend(loc='upper right', fontsize=10)
ax1.grid(True, alpha=0.3)
ax1.set_title('SPECTRA SED Fit', fontsize=14, fontweight='bold')
ax1.tick_params(labelbottom=False)

# Bottom panel: Residuals
residuals = (phot_data['obs_flux'] - results_ml['mod_flux']) / phot_data['obs_err']
ax2.scatter(wav_um, residuals, s=80, marker='o',
            facecolor='#2980B9', edgecolors='#1A5276', linewidth=1.5)
ax2.axhline(0, color='black', linestyle='-', linewidth=1)
ax2.axhspan(-1, 1, alpha=0.3, color='green')
ax2.axhline(1, color='green', linestyle='--', linewidth=0.8, alpha=0.8)
ax2.axhline(-1, color='green', linestyle='--', linewidth=0.8, alpha=0.8)

ax2.set_xscale('log')
ax2.set_xlabel('Wavelength [μm]', fontsize=12)
ax2.set_ylabel('Residual [σ]', fontsize=12)
ax2.set_ylim(-3, 3)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Print fit summary
print("\n" + "="*50)
print("BEST-FIT PARAMETERS")
print("="*50)
print(f"  log(M/Msun)  = {results_ml['parameters']['mass']:.2f}")
print(f
)
print(f"  [Z/H]        = {results_ml['parameters']['metallicity']:.2f}")
print(f"  E(B-V)       = {results_ml['parameters']['dust']:.2f}")
print("="*50)

# Compute chi-squared
chi2 = np.sum(residuals**2)
dof = len(phot_data['obs_flux']) - len(results_ml['parameters'])
chi2_red = chi2 / dof
print(f"\nχ²/DOF = {chi2_red:.2f}")
if chi2_red < 2:
    print("✓ Good fit!")
else:
    print("⚠ Poor fit - check data or model assumptions")

## 5. Export Results

In [ ]:
# Save results to DataFrame
results_df = pd.DataFrame({
    'wavelength_AA': phot_data['wavelength'],
    'band': phot_data['bands'],
    'obs_flux_Jy': phot_data['obs_flux'],
    'obs_err_Jy': phot_data['obs_err'],
    'model_flux_Jy': results_ml['mod_flux'],
    'residual_sigma': residuals
})

# Save to CSV
output_path = Path.home() / "spectra_results.csv"
results_df.to_csv(output_path, index=False)
print(f"✓ Results saved to: {output_path}")

# Display table
results_df

## 6. Batch Processing Example

Process multiple objects using config files.

In [ ]:
# Create config file for batch processing
import yaml

batch_config = {
    'input': {
        'type': 'rubin_tap',
        'ra': 150.0,
        'dec': 2.5,
        'radius_arcsec': 60.0  # Search 1 arcmin radius
    },
    'rubin': {
        'rsp_token': os.environ['RSP_TOKEN'],
        'flux_type': 'psfFlux',
        'bands': ['u', 'g', 'r', 'i', 'z', 'y']
    },
    'ssp_model': {
        'type': 'fsps',
        'imf': 'chabrier',
        'dust_type': 2
    },
    'fitting': {
        'method': 'ml',
        'error_floor': 0.05,
        'parameters': ['mass', 'age', 'metallicity', 'dust'],
        'priors': priors
    },
    'plotting': {
        'output_dir': 'outputs/rubin_batch',
        'save_plots': True
    },
    'output': {
        'save_photometry': True
    }
}

config_path = Path.home() / "spectra_batch.yaml"
with open(config_path, 'w') as f:
    yaml.dump(batch_config, f)

print(f"✓ Batch config saved to: {config_path}")
print("\nRun batch processing with:")
print(f"  spectra --config {config_path}")

## Summary

This notebook demonstrated:
1. ✓ Querying Rubin/LSST photometry via TAP
2. ✓ Running SPECTRA ML fitting (~1 second)
3. ✓ Visualizing SED fits and residuals
4. ✓ Exporting results to CSV
5. ✓ Setting up batch processing

**Next Steps:**
- Try MCMC fitting for full posteriors (uncomment cell 3b)
- Add external photometry (GALEX, WISE) via `additional_data`
- Process large batches with command-line tool
- Customize priors for your science case

**Documentation:**
- [Installation Guide](../docs/INSTALL.md)
- [Usage Guide](../docs/USAGE.md)
- [GitHub Repository](https://github.com/yourusername/SPECTRA)